# 42_07 - Multi-seed + CI + grafik (Paper 4-2, Fase 3)**Tujuan:** ulang perbandingan inti (5 aturan agregasi vs backdoor f=20%) pada5 seed {13,42,101,202,303}; laporkan rerata +/- sb + CI95 untuk MCC dan ASR;buat grafik ringkas. Pola kejujuran statistik warisi Paper 2/4.**Output:** `fed_out42/agg42.csv` + gambar `../figure-fed42/asr_by_rule.png`,`../figure-fed42/mcc_by_rule.png`.> Berat (5 seed x 5 rule). Jalankan setelah 42_01 di SageMaker. JANGAN mengarang.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scipy','scikit-learn'):
    nm=m if m!='scikit-learn' else 'sklearn'
    if importlib.util.find_spec(nm) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
sys.path.insert(0, os.path.abspath('.')); import robust_agg as ra
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
FIGDIR=os.path.join('..','figure-fed42'); os.makedirs(FIGDIR, exist_ok=True)
ROUNDS=60; LOCAL_EPOCHS=3; LR=3e-3; SEEDS=[13,42,101,202,303]
print('setup 42_07 siap')

## 1. Muat partisi + run_fed

In [ ]:
def need(path):
    assert os.path.exists(path),'PRASYARAT GAGAL: %s - jalankan 42_01 dulu'%path
    return path
d=np.load(need(os.path.join(OUTDIR,'parts_K10.npz')),allow_pickle=True)
X=np.asarray(d['X'],float); y=np.asarray(d['y'],int); parts=list(d['parts']); K=int(d['K'])
g=np.load(need(os.path.join(OUTDIR,'global_test.npz')),allow_pickle=True)
Xg=np.asarray(g['X'],float); yg=np.asarray(g['y'],int)
def predict_with(W,mu,sd):
    def f(Z):
        out,_=fed.forward(W, fed.zscore_apply(Z,mu,sd)); return (out>=0.5).astype(int)
    return f
def run_fed(rule='fedavg', attack=None, mal_ids=(), gamma=10.0, bd_frac=0.5, seed=42):
    cli=[]
    for k in range(K):
        ii=parts[k]; Xk=X[ii].copy(); yk=y[ii].copy(); rng=np.random.RandomState(seed+k)
        if attack=='label_flip' and k in mal_ids: yk=ra.attack_label_flip(yk,rng)
        if attack=='backdoor'  and k in mal_ids: Xk,yk=ra.poison_backdoor_data(Xk,yk,bd_frac,rng)
        if len(yk): mu,sd=fed.zscore_fit(Xk)
        else: mu,sd=np.zeros(X.shape[1]),np.ones(X.shape[1])
        cli.append(dict(X=fed.zscore_apply(Xk,mu,sd),y=yk,n=len(yk)))
    Wg=fed.init_weights(seed); f_byz=max(1,len(mal_ids)) if mal_ids else 1
    for t in range(ROUNDS):
        wl=[]; sizes=[]
        for k in range(K):
            if cli[k]['n']==0: continue
            Wl=fed.train_local(Wg,cli[k]['X'],cli[k]['y'],epochs=LOCAL_EPOCHS,lr=LR,mu=0.0,W_global=Wg,seed=seed)
            if attack=='scale' and k in mal_ids: Wl=ra.attack_scale_weights(Wl,gamma=gamma)
            wl.append(Wl); sizes.append(cli[k]['n'])
        Wg,_=ra.aggregate(rule,wl,sizes,f=f_byz,beta=max(1,f_byz))
    mu_g,sd_g=fed.zscore_fit(X); pf=predict_with(Wg,mu_g,sd_g); yp=pf(Xg)
    mcc=matthews_corrcoef(yg,yp)
    asr=ra.attack_success_rate(pf,Xg,yg) if attack=='backdoor' else np.nan
    return mcc,asr
RULES=['fedavg','krum','multi_krum','trimmed_mean','median']
print('K',K,'| seeds',SEEDS,'| run_fed siap')

## 2. 5 seed x 5 rule pada backdoor f=20% (MCC + ASR)

In [ ]:
mal=(0,1)
def ci95(a):
    a=np.asarray(a,float); m=a.mean(); s=a.std(ddof=1) if len(a)>1 else 0.0
    h=1.96*s/np.sqrt(len(a)) if len(a)>1 else 0.0; return m,s,h
rows=[]
for rule in RULES:
    mccs=[]; asrs=[]
    for sd in SEEDS:
        mcc,asr=run_fed(rule, attack='backdoor', mal_ids=mal, seed=sd)
        mccs.append(mcc); asrs.append(asr)
    mM,sM,hM=ci95(mccs); mA,sA,hA=ci95(asrs)
    rows.append({'rule':rule,'MCC_mean':round(mM,4),'MCC_std':round(sM,4),
                 'MCC_ci95':round(hM,4),'ASR_mean':round(mA,4),'ASR_std':round(sA,4),
                 'ASR_ci95':round(hA,4)})
    print(f'{rule:12s} MCC {mM:.4f}+/-{hM:.4f} | ASR {mA:.4f}+/-{hA:.4f}')
dfa=pd.DataFrame(rows); dfa.to_csv(os.path.join(OUTDIR,'agg42.csv'),index=False)
import IPython.display as ipd; ipd.display(dfa)
print('=== SEL 2 (multiseed agregasi) SELESAI ===')

## 3. Grafik (teks figur Inggris)

In [ ]:
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(dfa['rule'], dfa['ASR_mean'], yerr=dfa['ASR_ci95'], capsize=4, color='#c0504d')
ax.set_ylabel('Backdoor ASR'); ax.set_title('Backdoor ASR by aggregation (f=20%, 5 seeds)')
ax.set_ylim(0,1); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(os.path.join(FIGDIR,'asr_by_rule.png'),dpi=150); plt.close()
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(dfa['rule'], dfa['MCC_mean'], yerr=dfa['MCC_ci95'], capsize=4, color='#4f81bd')
ax.set_ylabel('MCC (GLOBAL-test)'); ax.set_title('MCC by aggregation under backdoor (f=20%, 5 seeds)')
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(os.path.join(FIGDIR,'mcc_by_rule.png'),dpi=150); plt.close()
print('grafik -> ../figure-fed42/{asr_by_rule,mcc_by_rule}.png')
print('=== SEL 3 (grafik) SELESAI ===')

## 4. Unggah hasil + catatan- `agg42.csv` + CSV nb 42_02-42_06 -> isi tabel `paper-4-2.tex` (angka NYATA).- Gambar `figure-fed42/*.png` (teks Inggris) -> figur paper.- Unggah ke S3 `unsw-far/federated42/results/` (boto3).

In [ ]:
try:
    import boto3; s3=boto3.client('s3',region_name='ap-southeast-1')
    B='ssh-detection-features-232032302717'; up=0
    for f in os.listdir(OUTDIR):
        if f.endswith('.csv'):
            s3.upload_file(os.path.join(OUTDIR,f),B,'unsw-far/federated42/results/'+f); up+=1
    for f in os.listdir(FIGDIR):
        if f.endswith('.png'):
            s3.upload_file(os.path.join(FIGDIR,f),B,'unsw-far/federated42/results/'+f); up+=1
    print('upload',up,'-> s3://%s/unsw-far/federated42/results/'%B)
except Exception as e: print('(upload S3 dilewati:',e,')')
print('=== SEL 4 (upload) SELESAI ===')